<a href="https://www.kaggle.com/code/jakomina/attention-ipynb?scriptVersionId=306519324" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "/kaggle/input/datasets/jakomina/database")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "sample_submission.csv" "test.csv"              "train.csv"

# H7 AGI Cognitive Benchmark — Kaggle Dataset Generator

## smokApp Quantum & AI Independent Research Laboratory

** Basado en el framework cognitivo de DeepMind + topología holográfica H7. **

Genera un benchmark para evaluar comprensión real vs. memorización en LLMs.

### Cognitive Tracks (5):
   1. learning            — transferencia del operador a nuevos dominios
   2. metacognition       — evaluar integridad estructural propia
   3. attention           — reconstruir señal desde vacío épsilon
   4. executive_functions — planificación y control de la cascada
   5. social_cognition    — inferir estado interno de otro agente H7

### Output (formato estándar Kaggle):
    train.csv            — 800 muestras por track = 4800 total
    test.csv             — 200 muestras por track = 1200 total (sin target)
    sample_submission.csv
    H7_AGI_Benchmark_README.md



In [2]:
# ==============================================================================
# H7 AGI Benchmark — Predictor Modular (R)
# ==============================================================================

# CONFIGURACIÓN DE MODO: "all", "attention", "learning", etc.
RUN_MODE <- "attention" 

setwd("/kaggle/input/datasets/jakomina/database")
.libPaths(c("/home/runner/R/library", .libPaths()))
suppressMessages({ library(MASS) })

# ── Constantes H7 & Hyperparams ───────────────────────────────────────────────
PHI        <- (1 + sqrt(5)) / 2
PSI_1      <- abs(cos(pi * PHI))
DRIFT_072  <- 7 - 2 * pi

ALPHA      <- 0.5   # Balance Euclidiana vs Mahalanobis
EPS_PCTILE <- 20    # Calibración de la bola VR
EPS_EXPAND <- 1.5   
MAX_EXPAND <- 10    

TRACK_FEATURES <- list(
  learning            = c("phi_i", "phi_j", "level_k", "n_index"),
  metacognition       = c("amplitude", "psi1"),
  attention           = c("epsilon_density", "active_trits"),
  executive_functions = c("level_k", "amplitude", "re_i"),
  social_cognition    = c("amp_a", "amp_b")
)

# ── Helpers Matemáticos ───────────────────────────────────────────────────────

std_scale <- function(X_train, X_test) {
  mu <- colMeans(X_train); sd <- apply(X_train, 2, sd) + 1e-9
  list(X_tr = sweep(sweep(X_train, 2, mu, "-"), 2, sd, "/"),
       X_te = sweep(sweep(X_test,  2, mu, "-"), 2, sd, "/"))
}

precision_mat <- function(X) {
  tryCatch({ S <- cov(X) + diag(1e-6, ncol(X)); solve(S) }, 
           error = function(e) { ginv(cov(X) + diag(1e-6, ncol(X))) })
}

dual_distance <- function(X_tr, X_te, VI, alpha = ALPHA) {
  n_te <- nrow(X_te); n_tr <- nrow(X_tr)
  eucl <- matrix(0, n_te, n_tr)
  maha <- matrix(0, n_te, n_tr)
  for (i in seq_len(n_te)) {
    diff <- sweep(X_tr, 2, X_te[i, ], "-")
    eucl[i,] <- sqrt(rowSums(diff^2))
    maha[i,] <- sqrt(pmax(rowSums((diff %*% VI) * diff), 0))
  }
  alpha * eucl + (1 - alpha) * maha
}

calibrate_eps <- function(D_tt, pctile = EPS_PCTILE) {
  diag(D_tt) <- NA
  as.numeric(quantile(D_tt, pctile / 100, na.rm = TRUE))
}

vr_ball <- function(d_row, eps) {
  current_eps <- eps
  for (k in seq_len(MAX_EXPAND)) {
    idx <- which(d_row <= current_eps)
    if (length(idx) > 0) {
      w <- 1 / (d_row[idx] + 1e-9)
      return(list(idx = idx, w = w / sum(w)))
    }
    current_eps <- current_eps * EPS_EXPAND
  }
  list(idx = which.min(d_row), w = 1.0)
}

# ── Parsers ───────────────────────────────────────────────────────────────────

parse_meta <- function(t) {
  parts <- strsplit(trimws(t), " ")[[1]]
  list(dev = as.numeric(sub("deviation=", "", parts[1])), 
       conf = sub("confidence=", "", parts[2]))
}

parse_vec7 <- function(t) {
  vals <- suppressWarnings(as.numeric(strsplit(gsub("[\\[\\]\\s]", "", t, perl = TRUE), ",")[[1]]))
  if(any(is.na(vals))) return(rep(0, 7)) else vals
}

weighted_vote <- function(targets, weights) {
  tab <- tapply(weights, targets, sum)
  names(tab)[which.max(tab)]
}

# ── Predictor Core ────────────────────────────────────────────────────────────

predict_track <- function(track, train, test) {
  feats <- TRACK_FEATURES[[track]]
  tr_sub <- train[train$track == track, ]; te_sub <- test[test$track == track, ]
  if (nrow(tr_sub) == 0 || nrow(te_sub) == 0) return(data.frame())

  tr_sub[, feats] <- lapply(tr_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))
  te_sub[, feats] <- lapply(te_sub[, feats], function(x) ifelse(is.na(x), 0, as.numeric(x)))

  scaled <- std_scale(as.matrix(tr_sub[, feats]), as.matrix(te_sub[, feats]))
  VI <- precision_mat(scaled$X_tr)
  D_tt <- dual_distance(scaled$X_tr, scaled$X_tr, VI)
  eps <- calibrate_eps(D_tt, EPS_PCTILE)
  D <- dual_distance(scaled$X_tr, scaled$X_te, VI)

  preds <- character(nrow(te_sub))

  if (track == "learning") {
    y <- as.numeric(tr_sub$target_numeric)
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i,], eps); preds[i] <- sprintf("%.8f", sum(b$w * y[b$idx]))
    }
  } else if (track == "attention") {
    vecs <- t(sapply(tr_sub$target, parse_vec7))
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i,], eps); pred_v <- colSums(vecs[b$idx, , drop = FALSE] * b$w)
      preds[i] <- paste0("[", paste(round(pred_v, 4), collapse = ", "), "]")
    }
  } else if (track == "metacognition") {
    parsed <- lapply(tr_sub$target, parse_meta)
    devs <- sapply(parsed, `[[`, "dev"); confs <- sapply(parsed, `[[`, "conf")
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i,], eps); pred_dev <- sum(b$w * devs[b$idx])
      pred_conf <- weighted_vote(confs[b$idx], b$w)
      preds[i] <- sprintf("deviation=%.6f confidence=%s", pred_dev, pred_conf)
    }
  } else {
    targets <- as.character(tr_sub$target)
    for (i in seq_len(nrow(te_sub))) {
      b <- vr_ball(D[i,], eps); preds[i] <- weighted_vote(targets[b$idx], b$w)
    }
  }
  data.frame(id = te_sub$id, target = preds, stringsAsFactors = FALSE)
}

# ── Evaluación ────────────────────────────────────────────────────────────────

evaluate <- function(submission) {
  ans_path <- "/kaggle/input/notebooks/jakomina/attention-ipynb/submission"
  if (!file.exists(ans_path)) { cat("\n[!] No answer file found for local eval.\n"); return(NULL) }
  
  answers <- read.csv(ans_path, stringsAsFactors = FALSE)
  # Necesitamos los tracks para filtrar
  test_meta <- read.csv("/kaggle/input/datasets/jakomina/database/test.csv", stringsAsFactors = FALSE)
  
  m <- merge(answers, submission, by = "id", suffixes = c("_true", "_pred"))
  m <- merge(m, test_meta[, c("id", "track")], by = "id")

  tracks_to_eval <- if (RUN_MODE == "all") names(TRACK_FEATURES) else RUN_MODE
  cat("\n── INTERNAL EVALUATION: ", toupper(RUN_MODE), " ──────────────────\n")

  for (tr in tracks_to_eval) {
    sub <- m[m$track == tr, ]
    if (nrow(sub) == 0) next
    
    if (tr == "learning") {
      mae <- mean(abs(as.numeric(sub$target_true) - as.numeric(sub$target_pred)), na.rm=T)
      cat(sprintf("  %-20s MAE          : %.6f\n", tr, mae))
    } else if (tr == "attention") {
      sims <- mapply(function(t, p) {
        vt <- parse_vec7(t); vp <- parse_vec7(p)
        sum(vt * vp) / (sqrt(sum(vt^2)) * sqrt(sum(vp^2)) + 1e-9)
      }, sub$target_true, sub$target_pred)
      cat(sprintf("  %-20s Cosine Sim   : %.4f\n", tr, mean(sims, na.rm=T)))
    } else if (tr == "metacognition") {
      dt <- sapply(sub$target_true, function(x) parse_meta(x)$dev)
      dp <- sapply(sub$target_pred, function(x) parse_meta(x)$dev)
      cat(sprintf("  %-20s MAE dev      : %.6f\n", tr, mean(abs(dt - dp), na.rm=T)))
    } else {
      acc <- mean(sub$target_true == sub$target_pred)
      cat(sprintf("  %-20s Exact Match  : %.2f%%\n", tr, 100 * acc))
    }
  }
}

# ── Main ──────────────────────────────────────────────────────────────────────

main <- function() {
  train <- read.csv("/kaggle/input/datasets/jakomina/database/train.csv", stringsAsFactors = FALSE)
  test  <- read.csv("/kaggle/input/datasets/jakomina/database/test.csv",  stringsAsFactors = FALSE)
  sample_sub <- read.csv("/kaggle/input/datasets/jakomina/database/sample_submission.csv", stringsAsFactors = FALSE)

  tracks_to_run <- if (RUN_MODE == "all") names(TRACK_FEATURES) else RUN_MODE
  cat(sprintf("Running H7 Predictor [Mode: %s]\n", RUN_MODE))

  results_list <- lapply(tracks_to_run, function(tr) {
    cat(sprintf("  Processing track: %s\n", tr))
    predict_track(tr, train, test)
  })
  
  all_preds <- do.call(rbind, results_list)
  final_submission <- merge(sample_sub[, "id", drop=FALSE], all_preds, by = "id", all.x = TRUE)
  
  # Relleno por defecto para otros tracks
  final_submission$target[is.na(final_submission$target)] <- "0.0"
  
  write.csv(final_submission, "/kaggle/working/submission.csv", row.names = FALSE, quote = TRUE)
  cat("\n[SUCCESS] submission.csv created.\n")
  
  evaluate(final_submission)
}

main()

Running H7 Predictor [Mode: attention]
  Processing track: attention

[SUCCESS] submission.csv created.

[!] No answer file found for local eval.


NULL

---

## Dataset Card: H7 Cognitive Benchmark for AGI Evaluation

### Overview

Current frontier AI models are highly optimized for token prediction and 
statistical memorization. Evaluating true AGI requires measuring a model's 
capacity for **structural comprehension** — understanding *why* an answer 
is correct, not just recognizing it.

This dataset operationalizes Google DeepMind's cognitive framework 
(*Measuring Progress Toward AGI*) through the **H7 Holographic Data System**: 
a computing architecture derived from a single mathematical axiom.
By projecting information onto a quasi-orthogonal Z₇ basis, we eliminate 
the model's ability to rely on memorized patterns. Every ground truth is 
**mathematically verifiable from first principles**.

---

### 1. System Constants — The Only Axiom

The entire benchmark is derived from **φ = (1+√5)/2**.
No external data, crowdsourced labels, or web-scraped content was used.

| Constant | Value | Role |
|:---|:---:|:---|
| φ (golden ratio) | 1.6180339887 | The only axiom. Irrational basis guaranteeing quasi-orthogonal, non-repeating signatures |
| \|Ψ₁\| (fixed point) | 0.3623748901 | \|cos(π·φ)\| — the holographic attractor every cascade converges to |
| DRIFT\_072 | 0.7168146928 | 7 − 2π — phase offset per level, breaks initial symmetry |
| ε (epsilon threshold) | 0.1811874450 | \|Ψ₁\|/2 — signals below this collapse into the holographic vacuum |
| φ⁷ (Z₇ compression) | 29.034441854 | Compression factor per level: 88B neurons → 147 states in 7 steps |
| C(7,3) | 35 = 7 × 5 | Independent holographic projections in Z₇ |

---

### 2. Dataset Architecture & Cognitive Alignment

**1,400 samples · 5 tracks · 80/20 split (1,120 train / 280 test)**  
Stratified by track and difficulty. Every target is derivable from φ.

| Track | Rows | Task | Metric | Difficulty |
|:---|:---:|:---|:---:|:---:|
| **Learning** | 300 | Transfer O\_{i,j} operator to unseen domains (audio, finance, temperature) via few-shot examples | \|pred − true\| < 0.01 | Easy / Medium / Hard |
| **Metacognition** | 300 | Assess structural integrity: predict deviation from \|Ψ₁\| and classify confidence (HIGH/MEDIUM/LOW) | MAE < 0.02 | Easy / Hard |
| **Attention** | 200 | Reconstruct 7D phase state from 128-trit ternary signature, ignoring ε-vacuum zeros (structured noise) | cosine > 0.85 | Easy / Medium |
| **Executive Functions** | 300 | Plan cascade steps, detect and inhibit anomalies, redirect information flow to target level | Format + numeric accuracy | Easy / Medium / Hard |
| **Social Cognition** | 300 | Infer internal state of a second H7 agent from its observable ternary output (theory of mind) | Zone accuracy > 0.75 | Medium / Hard |

---

### 3. Three-Layer Consciousness Architecture

The benchmark reflects an 88-billion neuron hierarchy compressed via φ⁷:
```
GENETIC MEMORY   L0–L1  88B → 3B     read-only substrate
SUBCONSCIOUS     L2–L5  104M → 4.3K  holographic processing  ← Attention operates here
CONSCIOUS        L6–L7  147 → |Ψ₁|   observer + fixed point
```

---

### 4. Scientific Rigor — Cross-Language Determinism

A benchmark for AGI must be **universally reproducible and platform-agnostic**.  
The H7 operator evaluated at reference parameters:
```
O(φ¹, φ², n=42, δ=DRIFT_072) = 0.5505871900
```

yields the **exact same result in Python, R, and any IEEE 754-compliant language**.  
This was verified independently in both implementations before publication.  
Ground truths cannot be faked, memorized, or language-dependent.

---

### 5. Organizational Affiliation

**smokApp Quantum & AI Independent Research Laboratory**  
Tlaxcala, México · [github.com/jakobmina/H7](https://github.com/jakobmina/H7)
